# Task 1: 셀 하나로 실행

Drive에 원본 **heloc_release.csv 또는 heloc_release.xlsx**를 올리세요.
코드 맨 위 **DATA_FILE**을 실제 Drive 경로로 바꾼 뒤 **코드 셀 하나만 실행**하세요.
처음 한 번 Drive 연결 승인이 필요합니다. 코드 ZIP을 업로드하지 않습니다.
예: `/content/drive/MyDrive/DeepLearning/heloc_release.csv`
결과는 원본 파일 옆의 `task1_results` 폴더에도 저장됩니다.
직접 업로드하고 싶다면 USE_DRIVE를 False로 바꾸세요. `(2)` 같은 파일명도 인식합니다.

후보 10개 × 학습 seed 2개 → Validation으로 단일 모델 선택 → Local Test 1회 평가 →
고정 설정으로 전체 데이터 재학습 → 학습 곡선과 결과 다운로드까지 자동 진행합니다.
여러 모델을 비교하므로 로그가 출력되는 동안 기다리세요.

전체 재학습을 생략하려면 코드 맨 위 REFIT_ALL_DATA만 False로 바꾸세요.
로컬 Test는 이전 실험에서 사용된 고객을 포함할 수 있으므로 역사적으로 완전히 미사용인 데이터는 아닙니다.
교수님 비공개 Test가 최종 평가입니다. 새 점수를 이전 0.7957과 직접 비교하지 마세요.

다운로드되는 ZIP은 **결과를 한 번에 보관하기 위한 파일**입니다. 실행을 위해 압축을 풀 필요는 없습니다.


In [ ]:
# Drive에 원본 CSV 또는 XLSX를 올리고 아래 DATA_FILE만 실제 경로로 바꾸세요.
# CPU로 실행합니다. 코드 ZIP 업로드나 개별 파일 실행은 필요 없습니다.
USE_DRIVE = True
DATA_FILE = '/content/drive/MyDrive/heloc_release.csv'
REFIT_ALL_DATA = True  # Local Test 평가 후, 고정 설정으로 전체 데이터 재학습
MODEL_SEEDS = [42, 43]
MAX_EPOCHS = 150
EARLY_STOP_PATIENCE = 20

import importlib.util
import subprocess
import sys

required = {'numpy': 'numpy', 'pandas': 'pandas', 'openpyxl': 'openpyxl',
            'sklearn': 'scikit-learn', 'joblib': 'joblib', 'torch': 'torch',
            'matplotlib': 'matplotlib'}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

"""HELOC DNN: compare on validation, evaluate held-out rows once, refit for submission.

python task1_complete.py fit --data heloc_release.xlsx --run results/run1
python task1_complete.py test --data heloc_release.xlsx --run results/run1
python task1_complete.py refit --data heloc_release.xlsx --run results/run1
python task1_complete.py predict --data new.xlsx --model results/run1/submission_model --out predictions.csv

Uses PyTorch and sklearn. Never runs training merely by importing this module.
"""
import argparse
import hashlib
import importlib.metadata
import json
from pathlib import Path
import random
import shutil

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.impute import KNNImputer
from sklearn.metrics import balanced_accuracy_score, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

CODES = (-7, -8, -9)
TARGET = 'is_at_risk'
IDENTIFIER = 'sample_id'


def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')


def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def read_data(path, labelled=True):
    df = pd.read_csv(path) if Path(path).suffix.lower() == '.csv' else pd.read_excel(path)
    required = [IDENTIFIER] + ([TARGET] if labelled else [])
    if not set(required).issubset(df.columns):
        raise ValueError(f'Required columns: {required}')
    if df.columns.duplicated().any() or df[IDENTIFIER].isna().any():
        raise ValueError('Duplicate column names or blank IDs.')
    df[IDENTIFIER] = df[IDENTIFIER].astype(str)
    if not df[IDENTIFIER].is_unique:
        raise ValueError('Duplicate IDs: define a grouped split before training.')
    if labelled:
        if df[TARGET].isna().any() or set(df[TARGET].unique()) != {0, 1}:
            raise ValueError('Training labels must contain both 0 and 1, with no blanks.')
        features = df.drop(columns=[IDENTIFIER, TARGET])
        if features.shape[1] != 23:
            raise ValueError('Expected the 23 original feature columns.')
        if features.duplicated().any():
            raise ValueError('Identical feature rows detected: review grouping before random splitting.')
    return df


def fingerprint(df):
    values = pd.util.hash_pandas_object(df, index=False).to_numpy().tobytes()
    return hashlib.sha256(json.dumps(list(df.columns)).encode()+values).hexdigest()


def fit_preprocessor(X, method):
    if X.isna().any().any():
        raise ValueError('Unexpected raw blank: define its meaning before training.')
    raw = X.to_numpy(dtype=float)
    if not np.isfinite(raw).all():
        raise ValueError('Input contains non-finite values.')
    observed = X.astype(float).mask(X.isin(CODES))
    if observed.isna().all().any():
        raise ValueError('A training feature has no observed numerical values.')
    scaler = StandardScaler().fit(observed)
    standardized = scaler.transform(observed)
    specs = [(c, int(code)) for c in X for code in CODES if X[c].eq(code).any()]
    imputer = None
    if method != 'zero':
        _, k, weighting = method.split('_')
        imputer = KNNImputer(n_neighbors=int(k), weights=weighting).fit(standardized)
    return dict(feature_names=list(X.columns), scaler=scaler, imputer=imputer,
                indicator_specs=specs, method=method)


def transform(X, state):
    missing = set(state['feature_names'])-set(X.columns)
    if missing:
        raise ValueError(f'Missing feature columns: {sorted(missing)}')
    raw = X.loc[:, state['feature_names']]
    if raw.isna().any().any() or not np.isfinite(raw.to_numpy(float)).all():
        raise ValueError('Unexpected blank or infinite raw value.')
    observed = raw.astype(float).mask(raw.isin(CODES))
    numeric = state['scaler'].transform(observed)
    if state['imputer'] is None:
        numeric = np.nan_to_num(numeric, nan=0.0)
    else:
        numeric = state['imputer'].transform(numeric)
    specs = state['indicator_specs']
    flags = np.column_stack([raw[c].eq(code).to_numpy(float) for c, code in specs]) if specs else np.empty((len(raw), 0))
    result = np.column_stack([numeric, flags]).astype(np.float32)
    if not np.isfinite(result).all():
        raise ValueError('Preprocessing produced non-finite values.')
    return result


class DNN(nn.Module):
    def __init__(self, n_features, config):
        super().__init__()
        layers = []
        previous = n_features
        for width in config['widths']:
            layers.append(nn.Linear(previous, width))
            if config['batchnorm']:
                layers.append(nn.BatchNorm1d(width))
            layers.extend([nn.ReLU(), nn.Dropout(config['dropout'])])
            previous = width
        layers.append(nn.Linear(previous, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)


def make_optimizer(model, config):
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim > 1 else no_decay).append(parameter)
    return torch.optim.AdamW([
        {'params': decay, 'weight_decay': config['weight_decay']},
        {'params': no_decay, 'weight_decay': 0.0},
    ], lr=config['lr'])


def make_loader(X, y, config, seed, batch_size):
    dataset = TensorDataset(torch.from_numpy(X), torch.tensor(y, dtype=torch.float32))
    # BatchNorm cannot estimate batch statistics from a singleton final batch.
    drop_last = config['batchnorm'] and len(dataset) % batch_size == 1
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0,
                      drop_last=drop_last, generator=torch.Generator().manual_seed(seed))


@torch.inference_mode()
def probabilities(model, X, device):
    model.eval()
    parts = []
    for start in range(0, len(X), 2048):
        logits = model(torch.from_numpy(X[start:start+2048]).to(device))
        parts.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(parts)


def one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.BCEWithLogitsLoss()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        if not torch.isfinite(loss):
            raise ValueError('Training loss is not finite.')
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()


def train_trial(tx, ty, vx, vy, config, seed, destination, args):
    destination.mkdir(parents=True, exist_ok=True)
    seed_everything(seed)
    model = DNN(tx.shape[1], config).to(args.device)
    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-5,
    )
    loader = make_loader(tx, ty, config, seed, args.batch_size)
    best_auc, best_epoch, best_state = -1.0, 0, None
    history = []
    for epoch in range(1, args.epochs+1):
        learning_rate = float(optimizer.param_groups[0]['lr'])
        one_epoch(model, loader, optimizer, args.device)
        vp = probabilities(model, vx, args.device)
        tp = probabilities(model, tx, args.device)
        score = float(roc_auc_score(vy, vp))
        history.append(dict(epoch=epoch, lr=learning_rate,
                            train_auroc=float(roc_auc_score(ty, tp)), valid_auroc=score,
                            train_bce=float(log_loss(ty, tp, labels=[0, 1])),
                            valid_bce=float(log_loss(vy, vp, labels=[0, 1]))))
        if score > best_auc:
            best_auc, best_epoch = score, epoch
            best_state = {k: value.detach().cpu().clone() for k, value in model.state_dict().items()}
        scheduler.step(score)
        if epoch-best_epoch >= args.patience:
            break
    checkpoint = dict(state_dict=best_state, n_features=tx.shape[1], config=config)
    torch.save(checkpoint, destination/'task1.pt')
    pd.DataFrame(history).to_csv(destination/'history.csv', index=False)
    result = dict(trial=destination.name, configuration=config['name'], seed=seed,
                  best_valid_auroc=best_auc, best_epoch=best_epoch, epochs_run=epoch)
    save_json(destination/'metrics.json', result)
    print(json.dumps(result), flush=True)
    return result


def candidates():
    def c(name, method='knn_30_uniform', widths=(64, 32), dropout=0.2, wd=1e-3, bn=False):
        return dict(name=name, method=method, widths=list(widths), dropout=dropout,
                    weight_decay=wd, batchnorm=bn, lr=1e-3)
    return [
        c('zero_baseline', 'zero', dropout=0.0, wd=1e-4),
        c('zero_regularized', 'zero'),
        c('knn15_regularized', 'knn_15_uniform'),
        c('knn30_regularized'),
        c('knn30_distance', 'knn_30_distance'),
        c('knn30_small', widths=(32, 16)),
        c('knn30_one_hidden', widths=(32,)),
        c('knn30_batchnorm', bn=True),
        c('knn30_stronger_regularization', dropout=0.35, wd=3e-3),
        c('zero_small', 'zero', widths=(32, 16)),
    ]


def load_model(directory, device):
    # Load only locally produced artifacts; joblib is not a safe untrusted-file format.
    checkpoint = torch.load(Path(directory)/'task1.pt', map_location='cpu', weights_only=True)
    model = DNN(checkpoint['n_features'], checkpoint['config']).to(device)
    model.load_state_dict(checkpoint['state_dict'])
    state = joblib.load(Path(directory)/'preprocessor.joblib')
    return model, state


def verified_dataset(args):
    root = Path(args.run)
    manifest = load_json(root/'split_manifest.json')
    df = read_data(args.data)
    if fingerprint(df) != manifest['dataset_fingerprint']:
        raise ValueError('Dataset differs from the one used to choose this model.')
    return root, manifest, df


def fit(args):
    root = Path(args.run)
    if root.exists() and any(root.iterdir()):
        raise ValueError('Use a new empty --run directory; previous results will not be overwritten.')
    root.mkdir(parents=True, exist_ok=True)
    df = read_data(args.data)
    X = df.drop(columns=[IDENTIFIER, TARGET])
    y = df[TARGET].to_numpy(dtype=np.int64)
    development, test = train_test_split(np.arange(len(df)), test_size=0.2,
                                         stratify=y, random_state=args.split_seed)
    train, valid = train_test_split(development, test_size=0.25,
                                   stratify=y[development], random_state=args.split_seed+1)
    manifest = dict(dataset_fingerprint=fingerprint(df), split_seed=args.split_seed,
                    train_ids=df.iloc[train][IDENTIFIER].tolist(),
                    valid_ids=df.iloc[valid][IDENTIFIER].tolist(),
                    test_ids=df.iloc[test][IDENTIFIER].tolist(),
                    note='Approximate 60/20/20. Prior experiments may have used these rows; this is a new procedural holdout, not guaranteed historically unseen data.')
    save_json(root/'split_manifest.json', manifest)
    print(f'Train={len(train)} Valid={len(valid)} Test={len(test)}; test metrics remain sealed.', flush=True)
    rows, cache = [], {}
    configs = candidates()
    for config in configs:
        method = config['method']
        if method not in cache:
            state = fit_preprocessor(X.iloc[train], method)
            # No transformation of held-out Test during candidate selection.
            tx, vx = transform(X.iloc[train], state), transform(X.iloc[valid], state)
            cache[method] = (state, tx, vx)
            joblib.dump(state, root/f'preprocessor_{method}.joblib')
        state, tx, vx = cache[method]
        for seed in args.seeds:
            destination = root/'trials'/f'{config["name"]}_seed{seed}'
            rows.append(train_trial(tx, y[train], vx, y[valid], config, seed, destination, args))
            pd.DataFrame(rows).to_csv(root/'trials.csv', index=False)
    table = pd.DataFrame(rows)
    summary = table.groupby('configuration').best_valid_auroc.agg(['mean', 'std', 'count']).reset_index()
    summary['std'] = summary['std'].fillna(0)
    summary = summary.sort_values(['mean', 'std'], ascending=[False, True])
    summary.to_csv(root/'validation_summary.csv', index=False)
    name = str(summary.iloc[0]['configuration'])
    chosen = table[table.configuration.eq(name)].sort_values(['best_valid_auroc', 'seed'], ascending=[False, True]).iloc[0]
    config = next(c for c in configs if c['name'] == name)
    trial = root/'trials'/str(chosen['trial'])
    shutil.copy2(trial/'task1.pt', root/'task1.pt')
    shutil.copy2(root/f'preprocessor_{config["method"]}.joblib', root/'preprocessor.joblib')
    history = pd.read_csv(trial/'history.csv')
    best_epoch = int(chosen['best_epoch'])
    decision = dict(config=config, seed=int(chosen['seed']), best_epoch=best_epoch,
                    batch_size=args.batch_size, lr_replay=history.lr.iloc[:best_epoch].tolist(),
                    selected_trial=str(chosen['trial']),
                    mean_valid_auroc=float(summary.iloc[0]['mean']),
                    selected_valid_auroc=float(chosen['best_valid_auroc']),
                    selection_rule='Highest mean validation AUROC across training seeds; one best validation checkpoint is kept. No ensemble.')
    save_json(root/'decision.json', decision)
    model, state = load_model(root, args.device)
    actual = probabilities(model, transform(X.iloc[valid], state), args.device)
    np.testing.assert_allclose(roc_auc_score(y[valid], actual), decision['selected_valid_auroc'], atol=1e-8)
    pd.DataFrame(dict(sample_id=df.iloc[valid][IDENTIFIER], label=y[valid], probability=actual)).to_csv(root/'validation_predictions.csv', index=False)
    (root/'requirements_actual.txt').write_text('\n'.join(
        f'{p}=={importlib.metadata.version(p)}' for p in ['numpy','pandas','openpyxl','scikit-learn','joblib','torch']
    )+'\n', encoding='utf-8')
    print('\nVALIDATION SUMMARY\n'+summary.to_string(index=False), flush=True)
    print('Selected '+name+'. Run the test command once; Test has not been scored.', flush=True)


def evaluate_test(args):
    root, manifest, df = verified_dataset(args)
    if (root/'test_report.json').exists():
        raise ValueError('This local Test has already been evaluated. Do not tune repeatedly on it.')
    if not (root/'decision.json').exists():
        raise ValueError('Complete candidate selection first.')
    rows = df.set_index(IDENTIFIER).loc[manifest['test_ids']].reset_index()
    model, state = load_model(root, args.device)
    p = probabilities(model, transform(rows, state), args.device)
    y = rows[TARGET].to_numpy(int)
    report = dict(local_test_rows=len(rows), local_test_auroc=float(roc_auc_score(y, p)),
                  local_test_bce=float(log_loss(y, p, labels=[0, 1])),
                  balanced_accuracy_at_05=float(balanced_accuracy_score(y, p >= 0.5)),
                  decision=load_json(root/'decision.json'),
                  caveat='One local evaluation after selection. Rows were used in earlier project exploration, so not a fully untouched final test. Instructor private Test is separate.')
    save_json(root/'test_report.json', report)
    pd.DataFrame(dict(sample_id=rows[IDENTIFIER], label=y, probability=p)).to_csv(root/'test_predictions.csv', index=False)
    print(json.dumps(report, indent=2), flush=True)


def refit(args):
    root, _, df = verified_dataset(args)
    if not (root/'test_report.json').exists():
        raise ValueError('Run the local Test once before refitting on all supplied rows.')
    destination = root/'submission_model'
    if destination.exists():
        raise ValueError('submission_model already exists; refusing to overwrite.')
    decision = load_json(root/'decision.json')
    config, seed = decision['config'], decision['seed']
    X = df.drop(columns=[IDENTIFIER, TARGET])
    state = fit_preprocessor(X, config['method'])
    tx = transform(X, state)
    seed_everything(seed)
    model = DNN(tx.shape[1], config).to(args.device)
    optimizer = make_optimizer(model, config)
    loader = make_loader(tx, df[TARGET].to_numpy(float), config, seed, decision['batch_size'])
    # Replay the frozen schedule up to the validation-selected epoch count.
    # No local Test metric or training-set score is used for model selection.
    for epoch, lr in enumerate(decision['lr_replay'], start=1):
        for group in optimizer.param_groups:
            group['lr'] = lr
        one_epoch(model, loader, optimizer, args.device)
        print(f'Full-data refit {epoch}/{decision["best_epoch"]}', flush=True)
    destination.mkdir(parents=True)
    torch.save(dict(state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
                    n_features=tx.shape[1], config=config), destination/'task1.pt')
    joblib.dump(state, destination/'preprocessor.joblib')
    save_json(destination/'training_info.json', dict(rows=len(df), decision=decision,
               note='Single DNN refit on all provided data. The local Test report belongs to the earlier held-out checkpoint, not this refitted model.'))
    # Verify a label-free inference round trip; not a training accuracy evaluation.
    probe = df.drop(columns=TARGET).iloc[:8]
    before = probabilities(model, transform(probe, state), args.device)
    restored, saved_state = load_model(destination, args.device)
    np.testing.assert_allclose(before, probabilities(restored, transform(probe, saved_state), args.device), rtol=1e-6, atol=1e-7)
    print('Saved submission_model/task1.pt and preprocessor.joblib; reload check passed.', flush=True)


def predict(args):
    df = read_data(args.data, labelled=False)
    model, state = load_model(args.model, args.device)
    p = probabilities(model, transform(df, state), args.device)
    destination = Path(args.out)
    destination.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(dict(sample_id=df[IDENTIFIER], probability=p)).to_csv(destination, index=False)
    print(f'Saved {len(df)} positive-class probabilities to {destination}', flush=True)


def parser():
    p = argparse.ArgumentParser(description=__doc__)
    commands = p.add_subparsers(dest='command', required=True)
    for name in ['fit', 'test', 'refit']:
        q = commands.add_parser(name)
        q.add_argument('--data', required=True)
        q.add_argument('--run', required=True)
        q.add_argument('--device', choices=['cpu','cuda'], default='cpu')
        if name == 'fit':
            q.add_argument('--seeds', type=int, nargs='+', default=[42,43])
            q.add_argument('--split-seed', type=int, default=20261125)
            q.add_argument('--epochs', type=int, default=150)
            q.add_argument('--patience', type=int, default=20)
            q.add_argument('--batch-size', type=int, default=128)
    q = commands.add_parser('predict')
    q.add_argument('--data', required=True)
    q.add_argument('--model', required=True)
    q.add_argument('--out', required=True)
    q.add_argument('--device', choices=['cpu','cuda'], default='cpu')
    return p



# ===================== 여기부터 자동 실행 =====================
from google.colab import files, drive
from datetime import datetime
from zoneinfo import ZoneInfo
import re
import matplotlib.pyplot as plt
from IPython.display import display

torch.set_num_threads(2)
if USE_DRIVE:
    drive.mount('/content/drive')
    data_path = Path(DATA_FILE)
    if data_path.suffix.lower() not in ['.csv', '.xlsx']:
        raise ValueError('DATA_FILE은 원본 .csv 또는 .xlsx 경로로 지정하세요.')
    if not data_path.is_file():
        raise FileNotFoundError(
            f'파일이 없습니다: {data_path}. Colab 왼쪽 파일 창에서 파일의 경로를 복사해 DATA_FILE에 넣으세요.'
        )
    results_root = data_path.parent / 'task1_results'
else:
    uploaded = files.upload()
    names = [name for name in uploaded
             if re.fullmatch(r'heloc_release(?: [(][0-9]+[)])?[.](?:csv|xlsx)', Path(name).name, flags=re.IGNORECASE)]
    if len(names) != 1:
        raise ValueError('heloc_release.csv 또는 heloc_release.xlsx 하나를 선택하세요.')
    data_path = Path('/content/task1_one_cell_input') / ('heloc_release'+Path(names[0]).suffix.lower())
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(uploaded[names[0]])
    results_root = Path('/content/task1_one_cell_results')
print('입력 파일:', data_path, flush=True)
run_id = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
run_dir = results_root / run_id
settings = argparse.Namespace(data=str(data_path), run=str(run_dir), device='cpu',
                              seeds=MODEL_SEEDS, split_seed=20261125,
                              epochs=MAX_EPOCHS, patience=EARLY_STOP_PATIENCE,
                              batch_size=128)
if MAX_EPOCHS < 1 or EARLY_STOP_PATIENCE < 1 or not MODEL_SEEDS:
    raise ValueError('epoch, patience와 최소 하나의 seed를 지정하세요.')

print('1/4: 후보 학습 및 Validation 선택. Local Test는 이 단계에서 사용하지 않습니다.', flush=True)
fit(settings)
display(pd.read_csv(run_dir / 'validation_summary.csv'))

decision = load_json(run_dir / 'decision.json')
history = pd.read_csv(run_dir / 'trials' / decision['selected_trial'] / 'history.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for key in ['train_auroc', 'valid_auroc']:
    axes[0].plot(history.epoch, history[key], label=key)
axes[0].axvline(decision['best_epoch'], color='gray', linestyle='--', label='selected epoch')
axes[0].set(xlabel='Epoch', ylabel='AUROC')
axes[0].legend()
for key in ['train_bce', 'valid_bce']:
    axes[1].plot(history.epoch, history[key], label=key)
axes[1].set(xlabel='Epoch', ylabel='BCE')
axes[1].legend()
fig.tight_layout()
fig.savefig(run_dir / 'learning_curves.png', dpi=150)
plt.show()

print('2/4: 선택된 단일 모델의 Local Test를 한 번 평가합니다.', flush=True)
evaluate_test(settings)
report = load_json(run_dir / 'test_report.json')

if REFIT_ALL_DATA:
    print('3/4: 설정·epoch·학습률 일정을 고정해 전체 데이터로 재학습합니다.', flush=True)
    refit(settings)
else:
    print('3/4: 전체 데이터 재학습을 생략했습니다.', flush=True)

print('4/4: 결과를 저장하고 다운로드합니다.', flush=True)
# Save the exact cell source, when available, for reproducibility.
try:
    from IPython import get_ipython
    cell_source = get_ipython().history_manager.input_hist_raw[-1]
    (run_dir / 'executed_colab_cell.py').write_text(cell_source, encoding='utf-8')
except (AttributeError, IndexError):
    pass

notes = f"""Task 1 결과
선택 설정: {decision['config']['name']}
Validation seed 평균 AUROC: {decision['mean_valid_auroc']:.6f}
선택한 단일 모델의 Validation AUROC: {decision['selected_valid_auroc']:.6f}
재학습 전 단일 모델의 Local Test AUROC: {report['local_test_auroc']:.6f}
전체 데이터 재학습: {REFIT_ALL_DATA}

test_report.json은 재학습 전 모델의 평가입니다.
submission_model은 전체 데이터로 재학습한 별도 단일 모델이며 새로운 Local Test 점수가 없습니다.
이전 실험에서 이미 해당 고객 데이터가 사용되었으므로 Local Test를 역사적으로 미사용인 평가라고 할 수 없습니다.
교수님 비공개 Test가 최종 평가입니다. Test 결과로 재튜닝하면 독립 평가가 아닙니다.
PyTorch 구현과 새 데이터 분리를 사용하므로 이전 NumPy AUROC 0.7957과 직접 비교하지 마세요.
가중치와 preprocessor.joblib을 함께 보관하세요. KNN 전처리에는 학습 이웃 정보가 있습니다.
"""
(run_dir / 'RESULT_NOTES.txt').write_text(notes, encoding='utf-8')
archive_path = shutil.make_archive(f'/content/task1_result_{run_id}', 'zip', root_dir=run_dir)
if USE_DRIVE:
    shutil.copy2(archive_path, results_root / Path(archive_path).name)
    print('Drive 결과 저장 위치:', run_dir, flush=True)
print(notes, flush=True)
files.download(archive_path)
